# LeetCode 794 — Valid Tic-Tac-Toe State bằng Python

Link đề: https://leetcode.com/problems/valid-tic-tac-toe-state/ · Độ khó: **Medium** · Chủ đề: Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def validTicTacToe(self, board: List[str]) -> bool:
        ...
```

Ghi chú: các ví dụ ngoài ba ví dụ của đề (ví dụ 4 trở đi ở mục 2) do tôi tự đặt, đáp án đã kiểm chứng bằng cách liệt kê **mọi trạng thái đạt được** của trò chơi; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho một bàn cờ **3 × 3** (cờ ca-rô nhỏ / tic-tac-toe) ở dạng 3 chuỗi, hãy cho biết bàn cờ đó **có thể xuất hiện** trong một ván chơi **hợp lệ** hay không (X đi trước, hai bên đi lần lượt, **ngừng ngay khi có người thắng**).
2. Nắm các công cụ Python: truy cập ô `board[i][j]` của list các chuỗi, `str.count`, `any`/`all`, danh sách 8 **đường thắng**, **biểu diễn bàn cờ bằng một chuỗi dài 9** hoặc bằng **mặt nạ bit**, hàng đợi `deque` để duyệt BFS và `lru_cache` để nhớ đệ quy.
3. Đi từ **liệt kê mọi trạng thái đạt được bằng BFS rồi tra cứu** → **kiểm tra bằng đếm quân và luật thắng** → **lùi nước đi cuối (retrograde)** → **bản mặt nạ bit**.
4. Hiểu mấu chốt: bàn cờ hợp lệ khi (a) số X bằng số O hoặc hơn đúng một; (b) nếu **X thắng** thì X vừa đi nước cuối, nên `số X = số O + 1`; (c) nếu **O thắng** thì O vừa đi nước cuối, nên `số X = số O`. (Hai bên cùng thắng sẽ tự bị loại bởi (b) và (c).)
5. Tránh các lỗi kinh điển: chỉ đếm quân mà quên điều kiện thắng, quên rằng X thắng bằng **hai đường cùng lúc** vẫn hợp lệ, bỏ sót một đường chéo, dùng `strip()` làm hỏng chỉ số, và cho phép O đi trước.

## 2. Đề bài & ví dụ

Cho bàn cờ ca-rô `board` là một mảng 3 chuỗi, mỗi chuỗi dài 3, gồm các ký tự `'X'`, `'O'` và `' '` (khoảng trắng = ô trống). Trả về `True` **khi và chỉ khi** có thể đạt tới bàn cờ này trong một ván tic-tac-toe **hợp lệ**.

Luật chơi:

- Hai người chơi lần lượt đặt quân vào các ô trống `' '`.
- Người đi trước **luôn đặt `'X'`**, người đi sau **luôn đặt `'O'`**.
- Chỉ được đặt vào **ô trống**, không được đặt đè.
- Trò chơi **kết thúc** khi có **ba ký tự giống nhau (khác trống)** thẳng hàng trong một hàng, một cột hoặc một đường chéo.
- Trò chơi cũng kết thúc khi **không còn ô trống**.
- **Khi trò chơi đã kết thúc thì không được đi thêm nước nào.**

**Ví dụ 1** (của đề)

- Input: `board = ["O  ","   ","   "]`
- Output: `false`
- Giải thích: người chơi đầu tiên luôn đặt `X`, nên không thể chỉ có một `O`.

**Ví dụ 2** (của đề)

- Input: `board = ["XOX"," X ","   "]`
- Output: `false`
- Giải thích: có 3 `X` nhưng chỉ 1 `O`: hai bên phải đi **luân phiên**, không thể có `X` nhiều hơn `O` hai quân.

**Ví dụ 3** (của đề)

- Input: `board = ["XOX","O O","XOX"]`
- Output: `true`

**Ví dụ 4 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `board = ["XXX","OO ","   "]`
- Output: `true`
- Giải thích: X đi 3 nước và thắng hàng trên; O đi 2 nước; X vừa đi nước cuối nên `X = O + 1`.

**Ví dụ 5 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `board = ["XXX","OOO","   "]`
- Output: `false`
- Giải thích: cả hai cùng có ba quân thẳng hàng; nhưng trò chơi dừng ngay khi người đầu tiên thắng, nên không thể có thêm nước đi của người kia.

**Ví dụ 6 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Input: `board = ["XXX","XOO","XOO"]`
- Output: `true`
- Giải thích: X thắng cùng lúc bằng **hai đường** (hàng trên và cột trái) nhờ nước cuối đặt vào góc trên-trái giao nhau. Có 5 `X` và 4 `O`, hợp lệ.

**Ràng buộc:**

- `board.length == 3`, `board[i].length == 3`.
- `board[i][j]` là `'X'`, `'O'` hoặc `' '`.

In [1]:
from typing import List
from collections import deque
from functools import lru_cache
from itertools import product
import random
import time

examples = [
    (["O  ", "   ", "   "], False),
    (["XOX", " X ", "   "], False),
    (["XOX", "O O", "XOX"], True),
    (["XXX", "OO ", "   "], True),
    (["XXX", "OOO", "   "], False),
    (["XXX", "XOO", "XOO"], True),
]


def show_board(board):
    for row in board:
        print("|" + "|".join(ch if ch != " " else "." for ch in row) + "|")


for board, expected in examples:
    print(board, "-> kỳ vọng", expected)
show_board(examples[3][0])

['O  ', '   ', '   '] -> kỳ vọng False
['XOX', ' X ', '   '] -> kỳ vọng False
['XOX', 'O O', 'XOX'] -> kỳ vọng True
['XXX', 'OO ', '   '] -> kỳ vọng True
['XXX', 'OOO', '   '] -> kỳ vọng False
['XXX', 'XOO', 'XOO'] -> kỳ vọng True
|X|X|X|
|O|O|.|
|.|.|.|


## 3. Kiến thức nền

### 3.1. Bàn cờ là list các chuỗi: `board[i][j]`

`board` có 3 phần tử, mỗi phần tử là một chuỗi 3 ký tự. `board[i]` là hàng `i`, và `board[i][j]` là ký tự ở **hàng `i`, cột `j`** (cả hai đánh số từ `0`). Ký tự khoảng trắng `' '` là ô trống — chú ý **đừng cắt khoảng trắng** (`strip()`), vì sẽ làm mất vị trí các ô.

In [2]:
board = ["XOX", " X ", "  O"]
print(board[0], "|", board[1][1], "|", repr(board[1][0]), "|", repr(board[2][2]))
print("các ô:", [(i, j, board[i][j]) for i in range(3) for j in range(3) if board[i][j] != " "])
print(repr("O  ".strip()))              # 'O' : mất 2 khoảng trắng!

XOX | X | ' ' | 'O'
các ô: [(0, 0, 'X'), (0, 1, 'O'), (0, 2, 'X'), (1, 1, 'X'), (2, 2, 'O')]
'O'


### 3.2. Đếm quân: `str.count` và `sum`

`row.count('X')` đếm số ký tự `'X'` trong chuỗi. Cộng qua 3 hàng được tổng số `X` trên bàn. Đây là phép đếm nền của bài.

In [3]:
board = ["XOX", "O O", "XOX"]
x = sum(row.count("X") for row in board)
o = sum(row.count("O") for row in board)
print("X =", x, ", O =", o)             # 5, 4
print("gộp thành 1 chuỗi:", "".join(board), "| đếm X:", "".join(board).count("X"))

X = 4 , O = 4
gộp thành 1 chuỗi: XOXO OXOX | đếm X: 4


### 3.3. Tám đường thắng

Có đúng **8 đường** ba ô thẳng hàng: 3 hàng, 3 cột, 2 đường chéo. Gom các ô `(i, j)` của từng đường vào một list để kiểm tra bằng vòng lặp. Với chuỗi dài 9 (`"".join(board)`), ô `(i, j)` có chỉ số `3 * i + j`, và 8 đường là các bộ ba `(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)`.

In [4]:
LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8),      # 3 hàng
         (0, 3, 6), (1, 4, 7), (2, 5, 8),      # 3 cột
         (0, 4, 8), (2, 4, 6)]                 # 2 đường chéo


def wins(s, ch):
    """s là chuỗi dài 9; ch có đủ ba ô thẳng hàng không?"""
    return any(s[a] == s[b] == s[c] == ch for a, b, c in LINES)


s = "".join(["XXX", "OO ", "   "])
print(wins(s, "X"), wins(s, "O"))                     # True False
print(wins("".join(["O X", " X ", "X O"]), "X"))      # True (đường chéo phụ)

True False
True


### 3.4. `any` và `all`

`any(iterable)` là `True` nếu **có ít nhất một** phần tử đúng; `all(iterable)` là `True` nếu **mọi** phần tử đúng. Chúng nhận cả **generator** nên không cần tạo list trung gian và dừng sớm ngay khi đủ kết luận. Ở đây `any(... for line in LINES)` hỏi "có đường thắng nào không?".

In [5]:
print(any(x > 2 for x in [1, 2, 3]), all(x > 2 for x in [1, 2, 3]))      # True False
print(any([]), all([]))                                                  # False True (rỗng)

True False
False True


### 3.5. Bàn cờ như một **trạng thái**; trạng thái đạt được

Mỗi bàn cờ là một trạng thái. Từ bàn trống, mỗi nước đi chuyển sang trạng thái mới. Một bàn cờ là **hợp lệ** nếu nó **nằm trong tập các trạng thái đạt được** từ bàn trống bằng các nước đi hợp lệ. Có thể tính tập này bằng BFS (duyệt theo tầng) và dùng làm "đáp án chuẩn" để kiểm chứng mọi cách khác.

### 3.6. Mặt nạ bit (bitmask)

Mỗi ô là một **bit**: ô `(i, j)` ứng với bit `3 * i + j`. Một số nguyên 9 bit `xm` biểu diễn tập các ô có `X`, `om` cho `O`. Một đường thắng là một mặt nạ `w` (ví dụ hàng trên `0b000000111`). Người chơi thắng nếu `xm & w == w` với một `w` nào đó. Số quân là **số bit 1**: `int.bit_count()` (Python 3.10+) hoặc `bin(x).count("1")`.

In [6]:
def to_masks(board):
    xm = om = 0
    for i in range(3):
        for j in range(3):
            if board[i][j] == "X":
                xm |= 1 << (3 * i + j)
            elif board[i][j] == "O":
                om |= 1 << (3 * i + j)
    return xm, om


WIN_MASKS = [sum(1 << k for k in line) for line in LINES]
xm, om = to_masks(["XXX", "OO ", "   "])
print(bin(xm), bin(om))
print([bin(w) for w in WIN_MASKS][:3])
print("X thắng:", any(xm & w == w for w in WIN_MASKS), "| số X:", xm.bit_count(), "| số O:", om.bit_count())

0b111 0b11000
['0b111', '0b111000', '0b111000000']
X thắng: True | số X: 3 | số O: 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy nghĩ về một ván chơi thật. X đi nước 1, O đi nước 2, X nước 3, ... Nên sau bất kỳ số nước nào:

- Nếu số nước chẵn: `X = O`. Nếu lẻ: `X = O + 1`. Vậy **`X = O` hoặc `X = O + 1`**, không có khả năng khác.

Nhưng chưa đủ, vì còn luật dừng khi có người thắng. Xét hai trường hợp:

- **X đã thắng.** Nước cuối cùng phải là nước của X (nếu O đi thêm sau khi X thắng thì vi phạm). Nước cuối của X ⇒ `X = O + 1`.
- **O đã thắng.** Nước cuối cùng là nước của O ⇒ `X = O`.

Thử trên ví dụ:

| bàn cờ | `X`, `O` | ai thắng | kết luận |
|---|---|---|---|
| `["O  ","   ","   "]` | 0, 1 | không | `X ≠ O` và `X ≠ O + 1` → **sai** |
| `["XOX"," X ","   "]` | 3, 1 | không | `X = O + 2` → **sai** |
| `["XOX","O O","XOX"]` | 5, 4 | không (không hàng nào đủ) | `X = O + 1` → **đúng** |
| `["XXX","OO ","   "]` | 3, 2 | X | `X = O + 1` ✓ → **đúng** |
| `["XXX","OOO","   "]` | 3, 3 | X và O | X thắng cần `X = O + 1` (không) → **sai** |

Cell dưới kiểm chứng bảng này bằng code ngắn.

In [7]:
def quick_check(board):
    s = "".join(board)
    x, o = s.count("X"), s.count("O")
    xw, ow = wins(s, "X"), wins(s, "O")
    ok = (x == o or x == o + 1) and not (xw and x != o + 1) and not (ow and x != o)
    return x, o, xw, ow, ok


for board, expected in examples:
    x, o, xw, ow, ok = quick_check(board)
    print(f"X={x} O={o} Xthắng={xw!s:<5} Othắng={ow!s:<5} -> {ok!s:<5} (kỳ vọng {expected})")
    assert ok == expected

X=0 O=1 Xthắng=False Othắng=False -> False (kỳ vọng False)
X=3 O=1 Xthắng=False Othắng=False -> False (kỳ vọng False)
X=4 O=4 Xthắng=False Othắng=False -> True  (kỳ vọng True)
X=3 O=2 Xthắng=True  Othắng=False -> True  (kỳ vọng True)
X=3 O=3 Xthắng=True  Othắng=True  -> False (kỳ vọng False)
X=5 O=4 Xthắng=True  Othắng=False -> True  (kỳ vọng True)


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi trạng thái đạt được (BFS) rồi tra cứu

**Ý tưởng.** Làm đúng như luật: xuất phát từ bàn trống (chuỗi 9 dấu cách), **thử mọi nước đi hợp lệ**:

1. Lấy một trạng thái `s` ra khỏi hàng đợi.
2. Nếu ván **đã kết thúc** (có người thắng hoặc hết ô trống) thì **không đi tiếp**.
3. Người đi tiếp theo là `X` nếu `số X == số O`, ngược lại là `O`.
4. Với mỗi ô trống, tạo trạng thái mới bằng cách đặt quân vào đó; nếu chưa gặp thì thêm vào tập `seen` và hàng đợi.

Cuối cùng `seen` chứa **mọi bàn cờ đạt được** (khoảng `5478` trạng thái). Với một `board` cho trước, trả lời bằng `"".join(board) in seen`.

In [8]:
def game_over(s):
    return wins(s, "X") or wins(s, "O") or " " not in s


def reachable_states():
    start = " " * 9
    seen = {start}
    queue = deque([start])
    while queue:
        s = queue.popleft()
        if game_over(s):
            continue                                   # đã kết thúc: không đi thêm
        player = "X" if s.count("X") == s.count("O") else "O"
        for i, c in enumerate(s):
            if c == " ":
                t = s[:i] + player + s[i + 1:]
                if t not in seen:
                    seen.add(t)
                    queue.append(t)
    return seen


REACH = reachable_states()
print("số trạng thái đạt được:", len(REACH))           # 5478


def valid_bfs(board):
    return "".join(board) in REACH


for board, expected in examples:
    got = valid_bfs(board)
    print(board, "->", got)
    assert got == expected

số trạng thái đạt được: 5478
['O  ', '   ', '   '] -> False
['XOX', ' X ', '   '] -> False
['XOX', 'O O', 'XOX'] -> True
['XXX', 'OO ', '   '] -> True
['XXX', 'OOO', '   '] -> False
['XXX', 'XOO', 'XOO'] -> True


**Độ phức tạp.** Liệt kê `S = 5478` trạng thái, mỗi trạng thái tạo tối đa 9 trạng thái con và kiểm tra thắng `O(8)`: tổng một lần cỡ `10^5` phép — **hằng số**, vì bàn cờ cố định 3 × 3. Sau đó mỗi truy vấn là `O(1)` (tra `set`). Bộ nhớ `O(S)`.

**Nút thắt.** Hoàn toàn đúng nhưng "nặng nề" cho một câu hỏi đơn giản; hơn nữa cách này không **giải thích** vì sao bàn cờ hợp lệ hay không. Ta nên rút ra các **điều kiện** thay vì liệt kê.

## 6. Ý tưởng thuật toán / bước đột phá

Quan sát từ mục 4: mọi bàn cờ đạt được thỏa **ba điều kiện cần**:

1. `X = O` hoặc `X = O + 1`.
2. Nếu X có đường thắng thì `X = O + 1`.
3. Nếu O có đường thắng thì `X = O`.

Điều đáng nói là chúng cũng **đủ** — tức là mọi bàn cờ thỏa ba điều kiện này đều có thể đạt được. Khẳng định này sẽ được chứng minh ở mục 10 và **kiểm chứng toàn diện** ở mục 12 (so với tập `REACH` trên cả `3^9 = 19683` bàn cờ).

Hai hệ quả tiện lợi: (a) không cần kiểm tra riêng "X và O cùng thắng", vì X thắng cần `X = O + 1` còn O thắng cần `X = O` — hai điều này không thể cùng đúng; (b) X có thể thắng bằng **hai đường cùng lúc** (hai đường chung ô vừa đặt) mà vẫn hợp lệ, nên **không** được loại trường hợp này.

## 7. Các cách tiếp theo

Tổng cộng 4 cách: cách 1 (BFS tra cứu, mục 5) và ba cách dưới đây.

### 7.1. Cách 2 — Đếm quân + kiểm tra đường thắng (chuẩn)

**Giải thích từng đoạn.**

- Đếm `x` = số `'X'`, `o` = số `'O'` trên cả bàn.
- Nếu `x != o` và `x != o + 1` → `False` (sai lượt).
- Hàm `win(ch)` kiểm tra 8 đường thắng của quân `ch`.
- Nếu `win('X')` mà `x != o + 1` → `False` (X thắng nhưng O đã đi tiếp, hoặc đếm sai).
- Nếu `win('O')` mà `x != o` → `False` (O thắng nhưng X đã đi tiếp).
- Còn lại → `True`.

In [9]:
def valid_rules(board):
    x = sum(row.count("X") for row in board)
    o = sum(row.count("O") for row in board)
    if x != o and x != o + 1:
        return False

    def win(ch):
        rows = any(all(board[i][j] == ch for j in range(3)) for i in range(3))
        cols = any(all(board[i][j] == ch for i in range(3)) for j in range(3))
        diag1 = all(board[k][k] == ch for k in range(3))
        diag2 = all(board[k][2 - k] == ch for k in range(3))
        return rows or cols or diag1 or diag2

    if win("X") and x != o + 1:
        return False
    if win("O") and x != o:
        return False
    return True


for board, expected in examples:
    got = valid_rules(board)
    print(board, "->", got)
    assert got == expected

['O  ', '   ', '   '] -> False
['XOX', ' X ', '   '] -> False
['XOX', 'O O', 'XOX'] -> True
['XXX', 'OO ', '   '] -> True
['XXX', 'OOO', '   '] -> False
['XXX', 'XOO', 'XOO'] -> True


**Code nộp LeetCode (cách 2):**

In [10]:
class Solution2:
    def validTicTacToe(self, board: List[str]) -> bool:
        x = sum(row.count("X") for row in board)
        o = sum(row.count("O") for row in board)
        if x != o and x != o + 1:
            return False

        def win(ch: str) -> bool:
            for i in range(3):
                if all(board[i][j] == ch for j in range(3)):
                    return True
                if all(board[j][i] == ch for j in range(3)):
                    return True
            if all(board[k][k] == ch for k in range(3)):
                return True
            return all(board[k][2 - k] == ch for k in range(3))

        if win("X") and x != o + 1:
            return False
        if win("O") and x != o:
            return False
        return True


for board, expected in examples:
    assert Solution2().validTicTacToe(board) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp cách 2.** Bàn cờ cố định 3 × 3 nên mọi thao tác là hằng số: thời gian **O(1)** (cụ thể cỡ vài chục phép so sánh), bộ nhớ **O(1)**. Nếu tổng quát hóa cho bàn `n × n` thì là `O(n²)`.

### 7.2. Cách 3 — Lùi nước đi cuối (retrograde / "undo")

**Ý tưởng.** Thay vì đi **tiến** từ bàn trống, ta đi **lùi** từ bàn đang xét: "bàn cờ này hợp lệ nếu có một nước đi cuối **hợp lệ** mà bỏ đi nó thì được một bàn cờ trước đó **hợp lệ**". Cụ thể với bàn `s`:

1. Bàn trống → hợp lệ.
2. Nếu `X = O` thì nước cuối là của `O`; nếu `X = O + 1` thì của `X`; còn lại → không hợp lệ.
3. Thử **gỡ từng quân** của người đi cuối. Bàn trước đó `prev` phải thỏa: **chưa có ai thắng** (nếu đã có người thắng thì không được đi thêm) và `prev` **hợp lệ** (đệ quy).
4. Có một cách gỡ nào thỏa thì hợp lệ.

Dùng `lru_cache` để không tính lại các bàn đã xét.

In [11]:
@lru_cache(maxsize=None)
def valid_undo_str(s):
    if s == " " * 9:
        return True
    x, o = s.count("X"), s.count("O")
    if x == o:
        last = "O"
    elif x == o + 1:
        last = "X"
    else:
        return False
    for i, c in enumerate(s):
        if c == last:
            prev = s[:i] + " " + s[i + 1:]
            if not wins(prev, "X") and not wins(prev, "O") and valid_undo_str(prev):
                return True
    return False


def valid_undo(board):
    return valid_undo_str("".join(board))


for board, expected in examples:
    got = valid_undo(board)
    print(board, "->", got)
    assert got == expected

['O  ', '   ', '   '] -> False
['XOX', ' X ', '   '] -> False
['XOX', 'O O', 'XOX'] -> True
['XXX', 'OO ', '   '] -> True
['XXX', 'OOO', '   '] -> False
['XXX', 'XOO', 'XOO'] -> True


**Code nộp LeetCode (cách 3):**

In [12]:
class Solution3:
    def validTicTacToe(self, board: List[str]) -> bool:
        def win(s: str, ch: str) -> bool:
            lines = ((0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6))
            return any(s[a] == s[b] == s[c] == ch for a, b, c in lines)

        @lru_cache(maxsize=None)
        def ok(s: str) -> bool:
            if s == " " * 9:
                return True
            x, o = s.count("X"), s.count("O")
            if x == o:
                last = "O"
            elif x == o + 1:
                last = "X"
            else:
                return False
            for i, c in enumerate(s):
                if c == last:
                    prev = s[:i] + " " + s[i + 1:]
                    if not win(prev, "X") and not win(prev, "O") and ok(prev):
                        return True
            return False

        return ok("".join(board))


for board, expected in examples:
    assert Solution3().validTicTacToe(board) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp cách 3.** Số bàn cờ khác nhau xét tới nhiều nhất là các tập con của các quân đã đặt: tối đa `2^9 = 512` trạng thái (mỗi ô có hoặc không có quân trong dãy lùi), mỗi trạng thái thử tối đa 9 cách gỡ: thời gian **O(1)** (hằng số lớn hơn cách 2), bộ nhớ **O(1)**. Cách này **trực tiếp mô phỏng định nghĩa** "đạt được", nên tiện cho việc chứng minh và kiểm chứng.

### 7.3. Cách 4 — Mặt nạ bit

**Ý tưởng.** Biểu diễn `X` và `O` bằng hai số nguyên 9 bit `xm`, `om` (mục 3.6). Đếm quân bằng `bit_count()`, kiểm tra thắng bằng `xm & w == w` với 8 mặt nạ `w`. Logic điều kiện **y hệt cách 2**, chỉ khác biểu diễn: nhanh, gọn, và hay gặp trong các bài trò chơi cần tốc độ.

In [13]:
def valid_bitmask(board):
    xm, om = to_masks(board)
    x, o = xm.bit_count(), om.bit_count()
    if x != o and x != o + 1:
        return False
    x_win = any(xm & w == w for w in WIN_MASKS)
    o_win = any(om & w == w for w in WIN_MASKS)
    if x_win and x != o + 1:
        return False
    if o_win and x != o:
        return False
    return True


for board, expected in examples:
    got = valid_bitmask(board)
    print(board, "->", got)
    assert got == expected

['O  ', '   ', '   '] -> False
['XOX', ' X ', '   '] -> False
['XOX', 'O O', 'XOX'] -> True
['XXX', 'OO ', '   '] -> True
['XXX', 'OOO', '   '] -> False
['XXX', 'XOO', 'XOO'] -> True


**Code nộp LeetCode (cách 4):**

In [14]:
class Solution4:
    def validTicTacToe(self, board: List[str]) -> bool:
        xm = om = 0
        for i in range(3):
            for j in range(3):
                if board[i][j] == "X":
                    xm |= 1 << (3 * i + j)
                elif board[i][j] == "O":
                    om |= 1 << (3 * i + j)
        wins = (0b000000111, 0b000111000, 0b111000000,        # hàng
                0b001001001, 0b010010010, 0b100100100,        # cột
                0b100010001, 0b001010100)                     # chéo
        x, o = bin(xm).count("1"), bin(om).count("1")
        if x != o and x != o + 1:
            return False
        if any(xm & w == w for w in wins) and x != o + 1:
            return False
        if any(om & w == w for w in wins) and x != o:
            return False
        return True


for board, expected in examples:
    assert Solution4().validTicTacToe(board) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp cách 4.** **O(1)** thời gian và bộ nhớ. Trong thực tế nhanh hơn cách 2 vài lần nhờ không duyệt lồng nhau, nhưng đề chỉ có một bàn cờ nên khác biệt không đáng kể.

## 8. TRACE từng bước

Ta trace trên `["XXX","OO ","   "]` (đáp án `True`) và `["XXX","OOO","   "]` (đáp án `False`).

### 8.1. Trace cách 2 (đếm quân + đường thắng)

Bảng trạng thái:

| Bước | Câu hỏi | Kết quả cần |
|---|---|---|
| 1 | `x`, `o` là bao nhiêu? | `x == o` hoặc `x == o + 1` |
| 2 | X có đường thắng nào? (liệt kê) | nếu có thì `x == o + 1` |
| 3 | O có đường thắng nào? (liệt kê) | nếu có thì `x == o` |

In [15]:
def trace_rules(board):
    show_board(board)
    s = "".join(board)
    x, o = s.count("X"), s.count("O")
    print(f"bước 1: x = {x}, o = {o} -> x == o? {x == o} | x == o + 1? {x == o + 1}")
    if x != o and x != o + 1:
        print("  -> sai lượt, False")
        return False
    names = {(0, 1, 2): "hàng 0", (3, 4, 5): "hàng 1", (6, 7, 8): "hàng 2",
             (0, 3, 6): "cột 0", (1, 4, 7): "cột 1", (2, 5, 8): "cột 2",
             (0, 4, 8): "chéo chính", (2, 4, 6): "chéo phụ"}
    ok = True
    for ch in "XO":
        lines = [names[l] for l in LINES if all(s[k] == ch for k in l)]
        print(f"bước {2 if ch == 'X' else 3}: đường thắng của {ch}: {lines or 'không có'}")
        if lines:
            need = (x == o + 1) if ch == "X" else (x == o)
            print(f"   {ch} thắng nên cần {'x == o + 1' if ch == 'X' else 'x == o'}: {need}")
            ok = ok and need
    print("kết quả:", ok)
    return ok

trace_rules(["XXX", "OO ", "   "])
print()
trace_rules(["XXX", "OOO", "   "])

|X|X|X|
|O|O|.|
|.|.|.|
bước 1: x = 3, o = 2 -> x == o? False | x == o + 1? True
bước 2: đường thắng của X: ['hàng 0']
   X thắng nên cần x == o + 1: True
bước 3: đường thắng của O: không có
kết quả: True

|X|X|X|
|O|O|O|
|.|.|.|
bước 1: x = 3, o = 3 -> x == o? True | x == o + 1? False
bước 2: đường thắng của X: ['hàng 0']
   X thắng nên cần x == o + 1: False
bước 3: đường thắng của O: ['hàng 1']
   O thắng nên cần x == o: True
kết quả: False


False

Quan sát: trường hợp thứ hai có `x = o = 3`, vừa X vừa O thắng; X thắng yêu cầu `x == o + 1` nhưng thực tế `x == o` → **sai**. Ta không cần câu hỏi riêng "hai bên cùng thắng": điều kiện của X và của O tự mâu thuẫn nhau.

### 8.2. Trace cách 3 (lùi nước đi cuối)

In từng bước lùi: nước cuối là của ai, thử gỡ ô nào, bàn trước đó có hợp lệ không:

In [16]:
def trace_undo(board):
    s = "".join(board)
    seen = set()

    def ok(s, depth=0):
        pad = "  " * depth
        view = "|".join(s[i:i + 3].replace(" ", ".") for i in (0, 3, 6))
        if s == " " * 9:
            print(f"{pad}[{view}] bàn trống -> hợp lệ")
            return True
        x, o = s.count("X"), s.count("O")
        last = "O" if x == o else "X" if x == o + 1 else None
        if last is None:
            print(f"{pad}[{view}] x = {x}, o = {o}: sai lượt -> không hợp lệ")
            return False
        print(f"{pad}[{view}] x = {x}, o = {o}: nước cuối là của {last}")
        for i, c in enumerate(s):
            if c != last:
                continue
            prev = s[:i] + " " + s[i + 1:]
            if wins(prev, "X") or wins(prev, "O"):
                print(f"{pad}  gỡ ô {i}: bàn trước đã có người thắng -> bỏ")
                continue
            print(f"{pad}  gỡ ô {i}: thử bàn trước")
            if ok(prev, depth + 2):
                return True
        print(f"{pad}  không cách gỡ nào hợp lệ -> không hợp lệ")
        return False

    return ok(s)

print("kết quả:", trace_undo(["XXX", "OO ", "   "]))

[XXX|OO.|...] x = 3, o = 2: nước cuối là của X
  gỡ ô 0: thử bàn trước
    [.XX|OO.|...] x = 2, o = 2: nước cuối là của O
      gỡ ô 3: thử bàn trước
        [.XX|.O.|...] x = 2, o = 1: nước cuối là của X
          gỡ ô 1: thử bàn trước
            [..X|.O.|...] x = 1, o = 1: nước cuối là của O
              gỡ ô 4: thử bàn trước
                [..X|...|...] x = 1, o = 0: nước cuối là của X
                  gỡ ô 2: thử bàn trước
                    [...|...|...] bàn trống -> hợp lệ
kết quả: True


Đọc trace: nước cuối là của `X` (vì `x = o + 1`). Gỡ `X` ở các ô `0`, `1` hay `2` (hàng trên) đều cho bàn trước đó **không có người thắng**, nên tiếp tục lùi; mỗi lần lùi xen kẽ `O`, `X`... cho tới bàn trống. Nếu gỡ một `X` **không thuộc đường thắng** (nếu có) thì bàn trước vẫn có đường thắng đã hoàn thành → bị loại.

### 8.3. Trace cách 4 (mặt nạ bit)

In hai mặt nạ dạng nhị phân 9 bit và từng phép `&`:

In [17]:
def trace_bitmask(board):
    xm, om = to_masks(board)
    print(f"xm = {xm:09b}  (số X = {xm.bit_count()})")
    print(f"om = {om:09b}  (số O = {om.bit_count()})")
    for w in WIN_MASKS:
        mark = ""
        if xm & w == w:
            mark += " <- X thắng"
        if om & w == w:
            mark += " <- O thắng"
        print(f"  w = {w:09b}: xm & w = {xm & w:09b}, om & w = {om & w:09b}{mark}")
    return valid_bitmask(board)

print("kết quả:", trace_bitmask(["XXX", "OO ", "   "]))

xm = 000000111  (số X = 3)
om = 000011000  (số O = 2)
  w = 000000111: xm & w = 000000111, om & w = 000000000 <- X thắng
  w = 000111000: xm & w = 000000000, om & w = 000011000
  w = 111000000: xm & w = 000000000, om & w = 000000000
  w = 001001001: xm & w = 000000001, om & w = 000001000
  w = 010010010: xm & w = 000000010, om & w = 000010000
  w = 100100100: xm & w = 000000100, om & w = 000000000
  w = 100010001: xm & w = 000000001, om & w = 000010000
  w = 001010100: xm & w = 000000100, om & w = 000010000
kết quả: True


Bit thứ `3 * i + j` ứng với ô `(i, j)`, đọc từ **bit thấp** (bên phải): hàng trên `XXX` là các bit `0, 1, 2` nên `xm = ...000000111`. Mặt nạ hàng trên `w = 000000111` cho `xm & w == w` → X thắng.

## 9. Code nộp LeetCode

Chọn **cách 2 (đếm quân + đường thắng)**: ngắn, đúng, dễ giải thích. Đây là code đúng class/method/signature (trên LeetCode `List` có sẵn; ở đây ta đã import ở trên):

In [18]:
class Solution:
    def validTicTacToe(self, board: List[str]) -> bool:
        x = sum(row.count("X") for row in board)
        o = sum(row.count("O") for row in board)

        def win(ch: str) -> bool:
            for i in range(3):
                if all(board[i][j] == ch for j in range(3)):        # hàng i
                    return True
                if all(board[j][i] == ch for j in range(3)):        # cột i
                    return True
            if all(board[k][k] == ch for k in range(3)):            # chéo chính
                return True
            return all(board[k][2 - k] == ch for k in range(3))     # chéo phụ

        if x != o and x != o + 1:        # hai bên đi luân phiên, X đi trước
            return False
        if win("X") and x != o + 1:      # X thắng -> X vừa đi nước cuối
            return False
        if win("O") and x != o:          # O thắng -> O vừa đi nước cuối
            return False
        return True


for board, expected in examples:
    got = Solution().validTicTacToe(board)
    print(board, "->", got)
    assert got == expected

['O  ', '   ', '   '] -> False
['XOX', ' X ', '   '] -> False
['XOX', 'O O', 'XOX'] -> True
['XXX', 'OO ', '   '] -> True
['XXX', 'OOO', '   '] -> False
['XXX', 'XOO', 'XOO'] -> True


## 10. Vì sao thuật toán đúng

Gọi `x`, `o` là số quân X, O; `XW`, `OW` là "X có đường thắng", "O có đường thắng". Ta chứng minh: **bàn cờ đạt được ⇔ (`x = o` hoặc `x = o + 1`) và (`XW ⇒ x = o + 1`) và (`OW ⇒ x = o`)**.

**(⇒) Điều kiện cần.** Xét một ván hợp lệ dẫn tới bàn cờ.

- Hai bên đi luân phiên, X trước ⇒ `x = o` hoặc `x = o + 1`.
- Nếu `XW`: đường thắng của X được hoàn thành **tại một nước đi của X**; sau đó ván dừng nên **không có thêm nước nào** — suy ra nước cuối cùng là của X ⇒ `x = o + 1`. (Nếu O đi thêm sau khi X đã thắng thì vi phạm luật dừng. Chú ý: X **có thể** hoàn thành hai đường cùng lúc, vẫn đúng.)
- Nếu `OW`: tương tự, nước cuối là của O ⇒ `x = o`.

**(⇐) Điều kiện đủ.** Giả sử các điều kiện thỏa. Ta **dựng** một ván hợp lệ bằng cách đi lùi (đúng cách 3):

- Nếu `x = o + 1` thì nước cuối là của X; nếu `x = o` thì của O.
- Nếu người đi cuối là **người thắng** (hoặc không ai thắng), chọn gỡ một quân thuộc **mọi đường thắng** của người đó — nếu họ có hai đường thắng, ta phải chọn quân ở **giao điểm** của hai đường. Với bàn cờ 3 × 3, hai đường thắng của cùng một người luôn **cắt nhau tại đúng một ô** (hai đường song song cần 6 quân cùng loại, nhưng khi đó `x ≥ 6` hoặc `o ≥ 6` mâu thuẫn với `x ≤ o + 1`, `x + o ≤ 9`), nên gỡ quân giao điểm làm mất cả hai đường.
- Bàn cờ trước đó **không còn đường thắng của người đi cuối**; và người kia **không có đường thắng** (nếu cả hai cùng có thì x = o + 1 và x = o cùng đúng — mâu thuẫn), nên ván trước đó chưa kết thúc — ta được phép đi tiếp.
- Số quân giảm một nên tiếp tục lùi (quy nạp theo số quân) tới bàn trống.

Việc dựng này bảo đảm mọi bàn thỏa điều kiện đều **đạt được**. Ta còn **kiểm chứng toàn diện** ở mục 12: so sánh điều kiện này với tập đạt được tính bằng BFS trên cả `3^9 = 19683` bàn cờ — khớp 100%.

**Cách 1** đúng theo định nghĩa. **Cách 3** đúng theo chứng minh đủ ở trên. **Cách 4** là cách 2 với biểu diễn khác.

## 11. Phân tích Time / Space Complexity

Bàn cờ cố định 3 × 3 nên mọi cách đều là hằng số theo kích thước đầu vào; bảng dưới so sánh cả khi tổng quát hóa cho bàn `n × n`.

| Cách | Time (3 × 3) | Time (n × n) | Space | Ghi chú |
|---|---|---|---|---|
| 1. BFS mọi trạng thái đạt được | O(1) mỗi truy vấn sau khi dựng `REACH` (≈ `10^5` phép một lần) | không mở rộng được (số trạng thái `3^(n²)`) | O(5478) | làm "chuẩn" kiểm chứng |
| 2. Đếm quân + đường thắng | O(1) | O(n²) | O(1) | **bản nộp** |
| 3. Lùi nước đi cuối | O(1), hằng số lớn | `2^(n²)` | O(1) (cache ≤ 512) | trực tiếp theo định nghĩa |
| 4. Mặt nạ bit | O(1) | O(n²) | O(1) | gọn và nhanh |

**Chi tiết cách 2.** Đếm quân: 9 ô. Kiểm tra thắng: 8 đường × 3 ô = 24 phép so sánh cho mỗi người chơi. Tổng cỡ vài chục phép — không phụ thuộc đầu vào.

## 12. Bộ test

Vì chỉ có `3^9 = 19683` bàn cờ có thể, ta có thể **kiểm tra vét cạn mọi đầu vào**. Các lớp test:

1. Ví dụ của đề (và ví dụ tự đặt) cho cả 4 cách + bản nộp.
2. Edge case: bàn trống, bàn đầy hòa, X thắng với hai đường, O thắng, cả hai thắng, các đường thắng ở cả 8 hướng.
3. **Phản ví dụ** của các cách làm sai.
4. **Kiểm tra vét cạn trên cả 19683 bàn cờ** với tập đạt được tính bằng một DFS mô phỏng ván chơi độc lập (viết khác BFS ở cách 1: đệ quy thật, bàn cờ dạng lưới).
5. Đo thời gian.

In [19]:
def oracle_states():
    """Oracle độc lập: DFS mô phỏng mọi ván chơi trên lưới 3x3 (list các list), không dùng chuỗi/LINES ở trên."""
    out = set()
    grid = [[" "] * 3 for _ in range(3)]

    def has_win(g, ch):
        for i in range(3):
            if g[i][0] == g[i][1] == g[i][2] == ch or g[0][i] == g[1][i] == g[2][i] == ch:
                return True
        return g[0][0] == g[1][1] == g[2][2] == ch or g[0][2] == g[1][1] == g[2][0] == ch

    def go(turn):
        out.add("".join("".join(r) for r in grid))
        if has_win(grid, "X") or has_win(grid, "O") or all(c != " " for r in grid for c in r):
            return
        for i in range(3):
            for j in range(3):
                if grid[i][j] == " ":
                    grid[i][j] = turn
                    go("O" if turn == "X" else "X")
                    grid[i][j] = " "

    go("X")
    return out


t0 = time.perf_counter()
ORACLE = oracle_states()
print("oracle DFS:", len(ORACLE), "trạng thái, mất", f"{(time.perf_counter() - t0):.2f}s")
assert ORACLE == REACH                   # hai cách liệt kê độc lập cho cùng tập
for board, expected in examples:
    assert ("".join(board) in ORACLE) == expected
print("oracle khớp ví dụ và khớp BFS của cách 1")

solvers = {
    "bfs": valid_bfs,
    "rules": valid_rules,
    "undo": valid_undo,
    "bitmask": valid_bitmask,
    "Solution2": lambda b: Solution2().validTicTacToe(b),
    "Solution3": lambda b: Solution3().validTicTacToe(b),
    "Solution4": lambda b: Solution4().validTicTacToe(b),
    "Solution": lambda b: Solution().validTicTacToe(b),
}
for board, expected in examples:
    for name, fn in solvers.items():
        assert fn(list(board)) == expected, (name, board)
print("tất cả cách khớp ví dụ:", list(solvers))

oracle DFS: 5478 trạng thái, mất 1.26s
oracle khớp ví dụ và khớp BFS của cách 1
tất cả cách khớp ví dụ: ['bfs', 'rules', 'undo', 'bitmask', 'Solution2', 'Solution3', 'Solution4', 'Solution']


In [20]:
# --- Edge case ---
def b3(s):
    return [s[0:3], s[3:6], s[6:9]]

edge_cases = {
    "bàn trống": b3(" " * 9),
    "một X": b3("X        "),
    "X, O": b3("XO       "),
    "hòa đầy (5X 4O)": ["XOX", "XOO", "OXX"],
    "X thắng hàng 0": b3("XXXOO    "),
    "X thắng hàng 1": b3("OO XXX  O"[:9]),
    "X thắng cột 0": b3("XOOXO X  "),
    "X thắng chéo chính": b3("XOOOXO  X"),
    "X thắng chéo phụ": b3("O XOX X O"[:9]),
    "X hai đường": ["XXX", "XOO", "XOO"],
    "O thắng hàng 2": b3("XX XX OOO"),
    "O thắng cột 2": b3("XXOX O  O"),
    "cả hai thắng": ["XXX", "OOO", "   "],
    "X thắng nhưng O đi tiếp": ["XXX", "OO ", "O  "],
    "O thắng nhưng X đi tiếp": ["OOO", "XX ", "XX "],
    "O đi trước": b3("O        "),
    "X hơn O hai quân": ["XOX", " X ", "   "],
    "O hơn X": ["OOX", "   ", "   "],
}
for title, board in edge_cases.items():
    assert all(len(r) == 3 for r in board) and len(board) == 3, title
    want = "".join(board) in ORACLE
    for name, fn in solvers.items():
        assert fn(list(board)) == want, (name, title, board)
    print(f"{title:<26}", board, "->", want)

bàn trống                  ['   ', '   ', '   '] -> True
một X                      ['X  ', '   ', '   '] -> True
X, O                       ['XO ', '   ', '   '] -> True
hòa đầy (5X 4O)            ['XOX', 'XOO', 'OXX'] -> True
X thắng hàng 0             ['XXX', 'OO ', '   '] -> True
X thắng hàng 1             ['OO ', 'XXX', '  O'] -> False
X thắng cột 0              ['XOO', 'XO ', 'X  '] -> False
X thắng chéo chính         ['XOO', 'OXO', '  X'] -> False
X thắng chéo phụ           ['O X', 'OX ', 'X O'] -> False
X hai đường                ['XXX', 'XOO', 'XOO'] -> True
O thắng hàng 2             ['XX ', 'XX ', 'OOO'] -> False
O thắng cột 2              ['XXO', 'X O', '  O'] -> True
cả hai thắng               ['XXX', 'OOO', '   '] -> False
X thắng nhưng O đi tiếp    ['XXX', 'OO ', 'O  '] -> False
O thắng nhưng X đi tiếp    ['OOO', 'XX ', 'XX '] -> False
O đi trước                 ['O  ', '   ', '   '] -> False
X hơn O hai quân           ['XOX', ' X ', '   '] -> False
O hơn X              

**Phản ví dụ.** Các cách "trông có vẻ đúng" nhưng sai:

- *Sai A:* chỉ kiểm tra đếm quân (`x == o` hoặc `x == o + 1`), bỏ qua luật thắng.
- *Sai B:* kiểm tra thắng nhưng dùng điều kiện sai: "X thắng ⇒ `x == o`".
- *Sai C:* loại bàn cờ khi **X có hai đường thắng** (cho là "thắng hai lần là bất thường").
- *Sai D:* bỏ sót **đường chéo phụ**.

In [21]:
def wrong_counts_only(board):
    x = sum(r.count("X") for r in board)
    o = sum(r.count("O") for r in board)
    return x == o or x == o + 1

def wrong_win_condition(board):
    s = "".join(board)
    x, o = s.count("X"), s.count("O")
    if x != o and x != o + 1:
        return False
    if wins(s, "X") and x != o:                 # SAI B: phải là x == o + 1
        return False
    if wins(s, "O") and x != o + 1:             # SAI B: phải là x == o
        return False
    return True

def wrong_two_lines(board):
    s = "".join(board)
    x, o = s.count("X"), s.count("O")
    if x != o and x != o + 1:
        return False
    x_lines = sum(all(s[k] == "X" for k in l) for l in LINES)
    o_lines = sum(all(s[k] == "O" for k in l) for l in LINES)
    if x_lines >= 2 or o_lines >= 2:            # SAI C: loại luôn hai đường thắng
        return False
    if x_lines and x != o + 1:
        return False
    if o_lines and x != o:
        return False
    return True

def wrong_no_anti_diagonal(board):
    s = "".join(board)
    x, o = s.count("X"), s.count("O")
    if x != o and x != o + 1:
        return False
    lines = LINES[:7]                           # SAI D: thiếu (2, 4, 6)
    xw = any(all(s[k] == "X" for k in l) for l in lines)
    ow = any(all(s[k] == "O" for k in l) for l in lines)
    if xw and x != o + 1:
        return False
    if ow and x != o:
        return False
    return True


tests = {
    "A": (wrong_counts_only, ["XXX", "OOO", "   "]),                    # đúng False, sai ra True
    "B": (wrong_win_condition, ["XXX", "OO ", "   "]),                  # đúng True, sai ra False
    "C": (wrong_two_lines, ["XXX", "XOO", "XOO"]),                      # đúng True, sai ra False
    "D": (wrong_no_anti_diagonal, ["O X", " XO", "X O"]),               # X thắng chéo phụ nhưng x == o -> đúng False
}
for label, (fn, board) in tests.items():
    truth = "".join(board) in ORACLE
    print(f"sai {label}: {fn(board)!s:<5} | đúng: {truth!s:<5} | {board}")
    assert fn(board) != truth

sai A: True  | đúng: False | ['XXX', 'OOO', '   ']
sai B: False | đúng: True  | ['XXX', 'OO ', '   ']
sai C: False | đúng: True  | ['XXX', 'XOO', 'XOO']
sai D: True  | đúng: False | ['O X', ' XO', 'X O']


Giải thích: **A** cho `True` ở `["XXX","OOO","   "]` vì `x = o = 3` thỏa đếm quân. **B** đảo điều kiện nên loại bàn hợp lệ `["XXX","OO ","   "]`. **C** loại nhầm `["XXX","XOO","XOO"]` — X đặt quân góc trên-trái hoàn thành **cùng lúc** hàng trên và cột trái. **D** bỏ sót đường chéo phụ (các ô `2, 4, 6`) nên không phát hiện X đã thắng trong khi O còn đi thêm.

**Kiểm tra vét cạn trên mọi bàn cờ.** Với `3^9 = 19683` bàn cờ, so sánh từng cách với tập `ORACLE` — nếu khớp tất cả thì ta đã chứng minh bằng máy rằng các cách đúng với **mọi** đầu vào của đề.

In [22]:
all_boards = ["".join(p) for p in product("XO ", repeat=9)]
print("tổng số bàn cờ khả dĩ:", len(all_boards))
valid_count = 0
for s in all_boards:
    board = [s[0:3], s[3:6], s[6:9]]
    want = s in ORACLE
    valid_count += want
    for name, fn in solvers.items():
        assert fn(list(board)) == want, (name, board, want)
print("số bàn hợp lệ:", valid_count)            # 5478
assert valid_count == 5478
print("8 cách khớp oracle DFS trên cả", len(all_boards), "bàn cờ (kiểm tra vét cạn)")

# các cách sai phải thật sự sai ở ít nhất một bàn cờ
for name, fn in (("A", wrong_counts_only), ("B", wrong_win_condition), ("C", wrong_two_lines), ("D", wrong_no_anti_diagonal)):
    bad = sum(1 for s in all_boards if fn([s[0:3], s[3:6], s[6:9]]) != (s in ORACLE))
    print(f"cách sai {name}: sai trên {bad} / {len(all_boards)} bàn cờ")
    assert bad > 0

tổng số bàn cờ khả dĩ: 19683


số bàn hợp lệ: 5478
8 cách khớp oracle DFS trên cả 19683 bàn cờ (kiểm tra vét cạn)
cách sai A: sai trên 568 / 19683 bàn cờ
cách sai B: sai trên 1354 / 19683 bàn cờ
cách sai C: sai trên 22 / 19683 bàn cờ
cách sai D: sai trên 71 / 19683 bàn cờ


In [23]:
# So sánh thời gian: mỗi cách xử lý toàn bộ 19683 bàn cờ
boards = [[s[0:3], s[3:6], s[6:9]] for s in all_boards]
for name in ("bfs", "rules", "undo", "bitmask", "Solution"):
    fn = solvers[name]
    t0 = time.perf_counter()
    for b in boards:
        fn(b)
    print(f"{name:>8}: {(time.perf_counter() - t0) * 1000:8.1f} ms cho {len(boards)} truy vấn")

     bfs:      3.8 ms cho 19683 truy vấn
   rules:     50.7 ms cho 19683 truy vấn
    undo:      5.0 ms cho 19683 truy vấn


 bitmask:     31.3 ms cho 19683 truy vấn
Solution:     46.7 ms cho 19683 truy vấn


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ kiểm tra số lượng quân

Đếm `x == o` hoặc `x == o + 1` là **cần** nhưng **chưa đủ**: còn luật "dừng khi có người thắng".

In [24]:
board = ["XXX", "OO ", "O  "]          # X thắng, rồi O đi tiếp (x = 3, o = 3)
print("sai  (chỉ đếm):", wrong_counts_only(board))
print("đúng           :", Solution().validTicTacToe(board))

sai  (chỉ đếm): True
đúng           : False


**Vì sao sai.** `x = o = 3` nên đếm quân vẫn cho `True`, nhưng X đã thắng ở hàng trên, và theo luật O **không được** đi tiếp. **Cách sửa:** thêm điều kiện thắng: nếu X thắng thì `x == o + 1`.

### Lỗi 2: đảo điều kiện thắng

Người nào thắng thì người đó vừa đi nước cuối: X thắng ⇒ `x == o + 1` (X đi lẻ), O thắng ⇒ `x == o` (O đi chẵn). Đảo hai điều kiện là lỗi hay gặp.

In [25]:
board = ["XXX", "OO ", "   "]
print("sai  (đảo điều kiện):", wrong_win_condition(board))
print("đúng                :", Solution().validTicTacToe(board))

sai  (đảo điều kiện): False
đúng                : True


**Vì sao sai.** Hàng trên `XXX` do X vừa đặt nước thứ 3, khi đó `x = 3`, `o = 2` nên `x = o + 1`. **Mẹo nhớ:** *người đi trước (X) kết thúc ở số quân lẻ; thắng ở nước cuối nghĩa là `x = o + 1`.*

### Lỗi 3: loại trường hợp X thắng bằng **hai đường cùng lúc**

Nước cuối có thể hoàn thành hai đường thẳng giao nhau tại ô vừa đặt; điều này hoàn toàn hợp lệ.

In [26]:
board = ["XXX", "XOO", "XOO"]
show_board(board)
print("sai  (loại 2 đường):", wrong_two_lines(board))
print("đúng               :", Solution().validTicTacToe(board))

|X|X|X|
|X|O|O|
|X|O|O|
sai  (loại 2 đường): False
đúng               : True


**Vì sao sai.** Bàn cờ có 5 X, 4 O; X đặt quân cuối ở góc trên-trái, nối cả hàng trên lẫn cột trái. **Cách sửa:** không đếm số đường thắng; chỉ hỏi "có thắng hay không".

### Lỗi 4: bỏ sót một đường chéo

Hai đường chéo là `(0,0),(1,1),(2,2)` và `(0,2),(1,1),(2,0)` — rất dễ quên đường thứ hai (`board[k][2 - k]`).

In [27]:
board = ["O X", " XO", "X O"]
show_board(board)
print("x, o =", sum(r.count("X") for r in board), sum(r.count("O") for r in board))
print("sai  (thiếu chéo phụ):", wrong_no_anti_diagonal(board))
print("đúng                 :", Solution().validTicTacToe(board))

|O|.|X|
|.|X|O|
|X|.|O|
x, o = 3 3
sai  (thiếu chéo phụ): True
đúng                 : False


**Vì sao sai.** X thắng ở đường chéo phụ nhưng `x = o = 3`: sau khi X thắng O vẫn đi thêm → không hợp lệ. Thiếu đường chéo phụ ta không nhận ra X đã thắng. **Cách sửa:** liệt kê đủ 8 đường (`LINES`) và dùng chung một hàm kiểm tra.

### Lỗi 5: dùng `strip()` hoặc `split()` làm mất ô trống

Ký tự trống `' '` là **dữ liệu**. Nếu bạn cắt khoảng trắng đầu/cuối mỗi hàng, `"O  "` thành `"O"` và các chỉ số `board[i][j]` hỏng.

In [28]:
board = ["O  ", "   ", "   "]
stripped = [row.strip() for row in board]
print(stripped)                                   # ['O', '', '']
try:
    print(stripped[0][1])
except IndexError as e:
    print("IndexError:", e)
print("đếm vẫn đúng nếu chỉ dùng count trên chuỗi gốc:", sum(r.count("O") for r in board))

['O', '', '']
IndexError: string index out of range
đếm vẫn đúng nếu chỉ dùng count trên chuỗi gốc: 1


**Vì sao sai.** `"O".strip()` chỉ còn 1 ký tự nên `stripped[0][1]` lỗi. **Cách sửa:** không `strip`; hoặc nối thành chuỗi 9 ký tự (`"".join(board)`) rồi truy cập theo chỉ số `3 * i + j`.

### Lỗi 6: cho phép O đi trước

Điều kiện lượt phải là `x == o` hoặc `x == o + 1` (X **không ít hơn** O). Nếu viết `abs(x - o) <= 1` thì bàn cờ `["O  ","   ","   "]` (ví dụ 1 của đề) bị coi là hợp lệ.

In [29]:
def wrong_abs(board):
    x = sum(r.count("X") for r in board)
    o = sum(r.count("O") for r in board)
    return abs(x - o) <= 1                       # SAI: cho phép o = x + 1

board = ["O  ", "   ", "   "]
print("sai  (abs <= 1):", wrong_abs(board))
print("đúng           :", Solution().validTicTacToe(board))
assert wrong_abs(board) != Solution().validTicTacToe(board)

sai  (abs <= 1): True
đúng           : False


**Vì sao sai.** X **luôn** đi trước nên `o` không thể lớn hơn `x`. **Cách sửa:** `x == o or x == o + 1`.

## 14. Mẹo & nhận dạng pattern

- **"Trạng thái này có thể đạt được từ trạng thái đầu bằng các nước đi hợp lệ?"** → tìm **bất biến/điều kiện cần** (số lượng, thứ tự lượt, điều kiện dừng), rồi chứng minh chúng **đủ** (đi lùi) và **kiểm chứng vét cạn** nếu không gian trạng thái nhỏ.
- **Bài trò chơi nhỏ** (3 × 3, vài nghìn trạng thái): có thể **liệt kê tất cả** bằng BFS/DFS để làm oracle; rất đáng làm.
- **Suy luận "ai đi nước cuối"** từ số quân; từ đó suy ra ai có thể thắng.
- **Trò chơi dừng khi thắng** ⇒ đừng quên ràng buộc "sau khi thắng không còn nước nào".
- **Danh sách đường thắng** (`LINES`) viết một lần, dùng chung cho cả hai người chơi.
- **Chọn biểu diễn hợp lý**: chuỗi dài 9, lưới, hoặc mặt nạ bit; biểu diễn tốt làm code ngắn và ít lỗi chỉ số.
- Họ hàng: **348 Design Tic-Tac-Toe**, **1275 Find Winner on a Tic Tac Toe Game**, các bài trạng thái hợp lệ như **36 Valid Sudoku**.

## 15. Biến thể

### 15.1. Trạng thái ván chơi: ai thắng, hòa, đang chơi, không hợp lệ

Dùng chính điều kiện hợp lệ rồi phân loại:

In [30]:
def game_status(board):
    if not Solution().validTicTacToe(board):
        return "invalid"
    s = "".join(board)
    if wins(s, "X"):
        return "X wins"
    if wins(s, "O"):
        return "O wins"
    return "draw" if " " not in s else "in progress"


print(game_status(["XXX", "OO ", "   "]))          # X wins
print(game_status(["XOX", "O O", "XOX"]))          # in progress
print(game_status(["XOX", "XOO", "OXX"]))          # draw
print(game_status(["O  ", "   ", "   "]))          # invalid
assert game_status(["XXX", "OO ", "   "]) == "X wins"
assert game_status(["XOX", "XOO", "OXX"]) == "draw"
assert game_status(["XOX", "O O", "XOX"]) == "in progress"
assert game_status(["XXX", "OOO", "   "]) == "invalid"

X wins
in progress
draw
invalid


### 15.2. Ai đi tiếp theo?

Nếu bàn hợp lệ và ván chưa kết thúc: `X` nếu `x == o`, ngược lại `O`.

In [31]:
def next_player(board):
    if game_status(board) != "in progress":
        return None
    s = "".join(board)
    return "X" if s.count("X") == s.count("O") else "O"

print(next_player(["XOX", "O O", "XOX"]))          # O (5 X, 4 O -> O đi)
print(next_player(["XXX", "OO ", "   "]))          # None (ván đã kết thúc)
print(next_player(["   ", "   ", "   "]))          # X
assert next_player(["   ", "   ", "   "]) == "X"
assert next_player(["X  ", "   ", "   "]) == "O"
assert next_player(["XXX", "OO ", "   "]) is None

X
None
X


### 15.3. Bàn cờ `n × n` với `k` quân liên tiếp (tổng quát hóa)

Cách làm kiểu "đếm quân + điều kiện thắng ⇒ ai đi cuối" vẫn dùng được, nhưng điều kiện **đủ** phức tạp hơn (hai đường thắng của cùng một người có thể không cắt nhau). Đây là gợi ý để tự suy nghĩ thêm; ta chỉ cài hàm kiểm tra thắng tổng quát:

In [32]:
def wins_general(grid, ch, k):
    n = len(grid)
    for i in range(n):
        for j in range(n):
            for di, dj in ((0, 1), (1, 0), (1, 1), (1, -1)):
                if all(0 <= i + t * di < n and 0 <= j + t * dj < n and grid[i + t * di][j + t * dj] == ch for t in range(k)):
                    return True
    return False

print(wins_general(["XXX", "OO ", "   "], "X", 3))           # True
print(wins_general(["XX ", "OO ", "   "], "X", 3))           # False
assert wins_general(["O X", " XO", "X O"], "X", 3)

True
False


### 15.4. Đếm số bàn cờ hợp lệ theo số quân

Dùng `REACH` để thống kê: có bao nhiêu trạng thái đạt được ứng với từng số nước đi.

In [33]:
from collections import Counter

by_moves = Counter(9 - s.count(" ") for s in REACH)
print(sorted(by_moves.items()))                     # [(0, 1), (1, 9), (2, 72), ...]
assert sum(by_moves.values()) == 5478
assert by_moves[0] == 1 and by_moves[1] == 9 and by_moves[2] == 72

[(0, 1), (1, 9), (2, 72), (3, 252), (4, 756), (5, 1260), (6, 1520), (7, 1140), (8, 390), (9, 78)]


### 15.5. Biến thể để tự nghĩ

- **Bàn cờ có thêm trạng thái "hòa" được xét riêng** (hết ô mà không ai thắng): thống kê số ván hòa hợp lệ.
- **Ai thắng "đáng ngờ"?** Một bàn hợp lệ có hai người cùng ba-quân-thẳng-hàng sẽ không xuất hiện; kiểm chứng bằng `REACH`.
- **Đếm số ván chơi khác nhau** (chuỗi nước đi) — `255168` ván đầy đủ; thử viết DFS đếm.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS mọi trạng thái đạt được | mô phỏng ván chơi, lưu tập trạng thái | dựng `~10^5`, mỗi truy vấn O(1) | O(5478) | làm oracle/kiểm chứng vét cạn |
| 2. Đếm quân + đường thắng | điều kiện cần-đủ từ lượt đi và luật dừng | O(1) | O(1) | **cách nộp mặc định** |
| 3. Lùi nước đi cuối | "gỡ" quân của người đi cuối, bàn trước phải hợp lệ và chưa kết thúc | O(1), hằng số lớn | O(1) | để chứng minh điều kiện đủ / bàn cờ lớn hơn |
| 4. Mặt nạ bit | cùng điều kiện, biểu diễn bằng 2 số 9 bit | O(1) | O(1) | khi cần tốc độ, hoặc bài nhiều truy vấn |

**Lời khuyên.** Nắm cách 2 (ba điều kiện), biết cách 1 như công cụ **kiểm chứng**, và cách 3 như công cụ **chứng minh**. Cách 4 là cải tiến biểu diễn.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [34]:
print(Solution().validTicTacToe(["XXX", "OO ", "   "]))
print(Solution().validTicTacToe(["XO ", "XO ", "XO "]))
print(Solution().validTicTacToe(["   ", "   ", "   "]))

True
False
True


**Bài 2 (tính tay).** Với `board = ["XOX", "OXO", "OXX"]`, đếm `X`, `O`, kiểm tra ai có đường thắng, rồi gán kết quả của `validTicTacToe` (một giá trị `bool`) vào `ans2` (đang là `None`).

In [35]:
ans2 = None      # TODO: True hoặc False

**Bài 3 (điền TODO).** Viết `has_won(board, ch)` trả về `True` nếu quân `ch` có **ba ô thẳng hàng** (hàng, cột hoặc chéo). `board` là list 3 chuỗi.

In [36]:
def has_won(board, ch):
    # TODO: kiểm tra 3 hàng, 3 cột, 2 đường chéo
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_valid(board)` theo **ba điều kiện** (đếm quân, X thắng, O thắng).

In [37]:
def my_valid(board):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) `count_valid_boards()` trả về **số bàn cờ hợp lệ** trong `3^9` bàn cờ, bằng cách gọi hàm kiểm tra của bạn trên mọi bàn (đáp án phải bằng `5478`). (b) `winning_cells(board)` trả về list các ô `(i, j)` thuộc **mọi đường thắng** của bàn cờ (rỗng nếu chưa ai thắng); với `["XXX","XOO","XOO"]` trả về `[(0,0),(0,1),(0,2),(1,0),(2,0)]` theo thứ tự quét hàng.

In [38]:
def count_valid_boards():
    # TODO
    pass

def winning_cells(board):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: ví dụ thứ hai có X thắng cột trái và O thắng cột phải (cả hai thắng).
- Bài 2: đếm cẩn thận từng hàng.
- Bài 3: 8 đường; dùng `all(...)` cho từng đường.
- Bài 4: ba `if` trả `False`, cuối cùng `True`.
- Bài 5: (a) dùng `itertools.product("XO ", repeat=9)`; (b) với mỗi đường thắng của X hoặc O, thu các ô của nó.

**Mức 2 (vừa):**
- Bài 2: `X = 5`, `O = 4`. Có hàng/cột/chéo nào toàn `X`? toàn `O`? (chéo chính là `X, X, X`).
- Bài 3: viết `LINES` thành list các bộ ba tọa độ `(i, j)`.
- Bài 4: `x != o and x != o + 1` → `False`; `win('X') and x != o + 1` → `False`; `win('O') and x != o` → `False`.
- Bài 5: (a) chuyển mỗi bộ chín ký tự thành 3 hàng; (b) gộp các ô vào `set`, rồi sắp xếp.

**Mức 3 (gần đáp án):**
- Bài 2: X có chéo chính nên cần `x == o + 1`: `5 == 4 + 1` ✓ → `True`.
- Bài 3: xem `win` trong mục 9.
- Bài 4: xem mục 9, bỏ chú thích.
- Bài 5: (a) `sum(valid(board) for ...)`; (b) `cells = set(); for line in LINES: if board[line[0]] == board[line[1]] == board[line[2]] != ' ': cells.update(line)` rồi đổi chỉ số thành `(i, j)` bằng `divmod(k, 3)`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [39]:
# Bài 1: X thắng, 3 X và 2 O -> True; hai cột cùng thắng (X cột 0, O cột 1) với 3 X, 3 O -> False; bàn trống -> True
assert Solution().validTicTacToe(["XXX", "OO ", "   "]) is True
assert Solution().validTicTacToe(["XO ", "XO ", "XO "]) is False
assert Solution().validTicTacToe(["   ", "   ", "   "]) is True

# Bài 2: ["XOX","OXO","OXX"]: X = 5, O = 4; X thắng chéo chính (X,X,X) với x = o + 1 -> True
ans2 = True
assert ans2 == Solution().validTicTacToe(["XOX", "OXO", "OXX"]) == (("XOXOXOOXX" in ORACLE))
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [40]:
# Bài 3
def has_won(board, ch):
    lines = [[(i, j) for j in range(3)] for i in range(3)]                 # hàng
    lines += [[(i, j) for i in range(3)] for j in range(3)]                # cột
    lines += [[(k, k) for k in range(3)], [(k, 2 - k) for k in range(3)]]  # chéo
    return any(all(board[i][j] == ch for i, j in line) for line in lines)

assert has_won(["XXX", "OO ", "   "], "X") is True
assert has_won(["XXX", "OO ", "   "], "O") is False
assert has_won(["O X", " XO", "X O"], "X") is True            # chéo phụ
assert has_won(["XO ", "XO ", "X  "], "X") is True            # cột 0
assert has_won(["   ", "   ", "   "], "X") is False
for s in all_boards[::7]:
    b = [s[0:3], s[3:6], s[6:9]]
    assert has_won(b, "X") == wins(s, "X") and has_won(b, "O") == wins(s, "O")
print("Bài 3 OK")

Bài 3 OK


In [41]:
# Bài 4
def my_valid(board):
    x = sum(r.count("X") for r in board)
    o = sum(r.count("O") for r in board)
    if x != o and x != o + 1:
        return False
    if has_won(board, "X") and x != o + 1:
        return False
    if has_won(board, "O") and x != o:
        return False
    return True

for s in all_boards:
    b = [s[0:3], s[3:6], s[6:9]]
    assert my_valid(b) == (s in ORACLE)
print("Bài 4 OK")

Bài 4 OK

In [42]:
# Bài 5 (a)
def count_valid_boards():
    total = 0
    for p in product("XO ", repeat=9):
        s = "".join(p)
        if my_valid([s[0:3], s[3:6], s[6:9]]):
            total += 1
    return total

assert count_valid_boards() == 5478

# Bài 5 (b)
def winning_cells(board):
    s = "".join(board)
    cells = set()
    for line in LINES:
        a, b, c = line
        if s[a] == s[b] == s[c] != " ":
            cells.update(line)
    return [divmod(k, 3) for k in sorted(cells)]

assert winning_cells(["XXX", "XOO", "XOO"]) == [(0, 0), (0, 1), (0, 2), (1, 0), (2, 0)]
assert winning_cells(["XOX", "O O", "XOX"]) == []
assert winning_cells(["O X", " XO", "X O"]) == [(0, 2), (1, 1), (2, 0)]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (ba điều kiện):**

```python
class Solution:
    def validTicTacToe(self, board: List[str]) -> bool:
        x = sum(row.count("X") for row in board)
        o = sum(row.count("O") for row in board)

        def win(ch):
            for i in range(3):
                if all(board[i][j] == ch for j in range(3)) or all(board[j][i] == ch for j in range(3)):
                    return True
            return all(board[k][k] == ch for k in range(3)) or all(board[k][2 - k] == ch for k in range(3))

        if x != o and x != o + 1:
            return False
        if win("X") and x != o + 1:
            return False
        if win("O") and x != o:
            return False
        return True
```

**Câu thần chú:** *X đi trước nên `x = o` hoặc `x = o + 1`; X thắng ⇒ `x = o + 1`; O thắng ⇒ `x = o`.*

**Checklist trước khi submit:**

- [ ] `x == o` hoặc `x == o + 1` (X không bao giờ ít hơn O)?
- [ ] X thắng ⇒ `x == o + 1`; O thắng ⇒ `x == o` (không đảo)?
- [ ] Đủ **8 đường thắng**, kể cả đường chéo phụ?
- [ ] Không loại trường hợp X thắng bằng **hai đường** cùng lúc?
- [ ] Không `strip()` các hàng (khoảng trắng là ô trống)?
- [ ] Đã thử: bàn trống, bàn đầy hòa, cả hai cùng thắng, O đi trước, ví dụ của đề?